In [1]:
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, f1_score)
from torchvision import transforms
from elpv_dataset.utils import load_dataset
import json

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {device}")

images, proba, types = load_dataset()
images_gray = images[:, :, :, np.newaxis]

def map_probability(p):
    valid_probs = np.array([0.0, 0.33, 0.67, 1.0])
    return valid_probs[np.argmin(np.abs(valid_probs - p))]

mapped_probs = np.array([map_probability(p) for p in proba])

class_mapping = {
    (0.00, 'mono'): 0, (0.00, 'poly'): 1, 
    (0.33, 'mono'): 2, (0.33, 'poly'): 3, 
    (0.67, 'mono'): 2, (0.67, 'poly'): 3, 
    (1.00, 'mono'): 4, (1.00, 'poly'): 5
}
labels_6class = np.array([class_mapping[(float(p), t)] for p, t in zip(mapped_probs, types)])
material_mapping = {'mono': 0, 'poly': 1}
labels_material = np.array([material_mapping[t] for t in types])

ordinal_targets = []
for p in mapped_probs:
    ordinal_targets.append([
        1.0 if p >= 0.33 else 0.0,
        1.0 if p >= 1.00 else 0.0
    ])
ordinal_targets = np.array(ordinal_targets, dtype=np.float32)

indices = np.arange(len(images_gray))
idx_train, idx_temp = train_test_split(indices, test_size=0.30, stratify=labels_6class, random_state=SEED)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, stratify=labels_6class[idx_temp], random_state=SEED)

class OrdinalSolarDataset(Dataset):
    def __init__(self, images, labels_6c, ordinal_targets, labels_mat, indices, transform=None):
        self.images = images[indices]
        self.labels_6c = labels_6c[indices]
        self.ordinal_targets = ordinal_targets[indices]
        self.labels_mat = labels_mat[indices]
        self.transform = transform
        
    def __len__(self):
        return len(self.images)
        
    def __getitem__(self, idx):
        img = self.transform(self.images[idx])
        return img, {
            'label_6c': torch.tensor(self.labels_6c[idx], dtype=torch.long),
            'ordinal_logits': torch.tensor(self.ordinal_targets[idx], dtype=torch.float32),
            'label_mat': torch.tensor(self.labels_mat[idx], dtype=torch.long)
        }

train_transforms = transforms.Compose([
    transforms.ToPILImage(), transforms.Resize((120, 120)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(), transforms.RandomRotation(15),
    transforms.ToTensor(), transforms.Normalize(mean=[0.5], std=[0.5])
])

test_transforms = transforms.Compose([
    transforms.ToPILImage(), transforms.Resize((120, 120)),
    transforms.ToTensor(), transforms.Normalize(mean=[0.5], std=[0.5])
])

train_dataset = OrdinalSolarDataset(images_gray, labels_6class, ordinal_targets, labels_material, idx_train, train_transforms)
val_dataset = OrdinalSolarDataset(images_gray, labels_6class, ordinal_targets, labels_material, idx_val, test_transforms)
test_dataset = OrdinalSolarDataset(images_gray, labels_6class, ordinal_targets, labels_material, idx_test, test_transforms)

train_labels_for_sampler = labels_6class[idx_train]
class_counts = np.bincount(train_labels_for_sampler, minlength=6)
class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = torch.DoubleTensor([class_weights[label] for label in train_labels_for_sampler])
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

class LabelSmoothingBCEWithLogitsLoss(nn.Module):
    def __init__(self, smoothing=0.1):
        super(LabelSmoothingBCEWithLogitsLoss, self).__init__()
        self.smoothing = smoothing
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        smoothed_targets = targets * (1.0 - self.smoothing) + 0.5 * self.smoothing
        return self.bce(logits, smoothed_targets)

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.leaky = nn.LeakyReLU(0.1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.leaky(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = self.leaky(out)
        return out

class DecoupledMultiScaleFusionCNN(nn.Module):
    def __init__(self):
        super(DecoupledMultiScaleFusionCNN, self).__init__()
        self.init_conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.1)
        )
        
        self.layer1 = ResidualBlock(32, 32, stride=1)
        self.pool1 = nn.MaxPool2d(2, 2) 
        
        self.layer2 = ResidualBlock(32, 64, stride=2)   
        self.layer3 = ResidualBlock(64, 128, stride=2)  
        
        self.local_track = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1)
        )
        self.context_track = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, padding=2, dilation=2, bias=False),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1)
        )
        
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.flatten = nn.Flatten()
        
        self.defect_head = nn.Sequential(
            nn.Linear(128, 64),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.3),
            nn.Linear(64, 2) 
        )
        
        self.material_head = nn.Sequential(
            nn.Linear(128, 48),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.3),
            nn.Linear(48, 2) 
        )

    def forward(self, x):
        x = self.init_conv(x)
        x = self.pool1(self.layer1(x))
        x = self.layer2(x)
        x = self.layer3(x)
        
        feat_local = self.local_track(x)
        feat_context = self.context_track(x)
        fused = torch.cat([feat_local, feat_context], dim=1)
        
        shared_features = self.flatten(self.global_pool(fused))
        
        ord_logits = self.defect_head(shared_features)
        mat_logits = self.material_head(shared_features)
        return ord_logits, mat_logits

model = DecoupledMultiScaleFusionCNN().to(device)

criterion_ls_bce = LabelSmoothingBCEWithLogitsLoss(smoothing=0.1)
criterion_ce = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4, min_lr=1e-6)

EPOCHS, PATIENCE = 200, 15
best_val_loss = float("inf")
patience_counter = 0
best_model_state = None

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for images_batch, targets_batch in train_loader:
        images_batch = images_batch.to(device)
        t_ord = targets_batch['ordinal_logits'].to(device)
        t_mat = targets_batch['label_mat'].to(device)
        
        optimizer.zero_grad()
        pred_ord, pred_mat = model(images_batch)
        
        violation_penalty = torch.relu(pred_ord[:, 1] - pred_ord[:, 0]).mean()
        
        loss = criterion_ls_bce(pred_ord, t_ord) + criterion_ce(pred_mat, t_mat) + 0.5 * violation_penalty
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for images_batch, targets_batch in val_loader:
            images_batch = images_batch.to(device)
            t_ord = targets_batch['ordinal_logits'].to(device)
            t_mat = targets_batch['label_mat'].to(device)
            pred_ord, pred_mat = model(images_batch)
            val_loss += (criterion_ls_bce(pred_ord, t_ord) + criterion_ce(pred_mat, t_mat)).item()
            
    train_loss /= len(train_loader)
    val_loss /= len(val_loader)
    
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr'] 
    print(f"Epoch: [{epoch+1:02d}/{EPOCHS}] Training Loss: {train_loss:.4f} | Validation Loss: {val_loss:.4f} | LR: {current_lr:.2e}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early Stopping Triggered at Epoch {epoch+1}")
            break

if best_model_state is not None:
    model.load_state_dict(best_model_state)
model = model.to(device)
model.eval()

def extract_raw_predictions(loader):
    raw_ord, raw_mat, true_6c = [], [], []
    with torch.no_grad():
        for images_batch, targets_batch in loader:
            images_batch = images_batch.to(device)
            pred_ord, pred_mat = model(images_batch)
            
            raw_ord.append(torch.sigmoid(pred_ord).cpu().numpy())
            raw_mat.append(torch.softmax(pred_mat, dim=1).cpu().numpy())
            true_6c.extend(targets_batch['label_6c'].numpy())
            
    return np.vstack(raw_ord), np.vstack(raw_mat), np.array(true_6c)

val_ord_raw, val_mat_softmax, y_val_true = extract_raw_predictions(val_loader)
test_ord_raw, test_mat_softmax, y_test_true = extract_raw_predictions(test_loader)

best_macro_f1 = -1
best_th_medium = 0.5
best_th_high = 0.5

for th_med in np.linspace(0.35, 0.65, 7):
    for th_high in np.linspace(0.35, 0.65, 7):
        
        v_ord = np.zeros_like(val_ord_raw)
        v_ord[:, 0] = val_ord_raw[:, 0]
        v_ord[:, 1] = np.minimum(val_ord_raw[:, 0], val_ord_raw[:, 1])
        
        p_med_triggered = (v_ord[:, 0] >= th_med).astype(float)
        p_high_triggered = (v_ord[:, 1] >= th_high).astype(float)
        
        v_probs = np.zeros((len(val_ord_raw), 3))
        v_probs[:, 0] = 1.0 - p_med_triggered
        v_probs[:, 1] = p_med_triggered - p_high_triggered
        v_probs[:, 2] = p_high_triggered
        
        val_defect_class = np.argmax(v_probs, axis=1)
        val_mat_class = np.argmax(val_mat_softmax, axis=1)
        y_val_pred = (val_defect_class * 2) + val_mat_class
        
        score = f1_score(y_val_true, y_val_pred, average='macro', zero_division=0)
        if score > best_macro_f1:
            best_macro_f1 = score
            best_th_medium = th_med
            best_th_high = th_high

print(f"Optimal Validation Thresholds Found -> Medium: {best_th_medium:.2f} | High: {best_th_high:.2f}")

p_ord_test = np.zeros_like(test_ord_raw)
p_ord_test[:, 0] = test_ord_raw[:, 0]
p_ord_test[:, 1] = np.minimum(test_ord_raw[:, 0], test_ord_raw[:, 1])

test_med_triggered = (p_ord_test[:, 0] >= best_th_medium).astype(float)
test_high_triggered = (p_ord_test[:, 1] >= best_th_high).astype(float)

reconstituted_test_probs = np.zeros((len(test_ord_raw), 3))
reconstituted_test_probs[:, 0] = 1.0 - test_med_triggered
reconstituted_test_probs[:, 1] = test_med_triggered - test_high_triggered
reconstituted_test_probs[:, 2] = test_high_triggered

test_defect_class = np.argmax(reconstituted_test_probs, axis=1)
test_mat_class = np.argmax(test_mat_softmax, axis=1)
y_test_pred = (test_defect_class * 2) + test_mat_class

print("\n================ FINAL REPORT ================")
acc = accuracy_score(y_test_true, y_test_pred)
f1_macro = f1_score(y_test_true, y_test_pred, average='macro')
print(f"Accuracy: {acc * 100:.2f}%")
print(f"Macro F1-Score:        {f1_macro:.4f}\n")

target_names = [
    '0.00 Mono', '0.00 Poly', 
    '0.33 & 0.66 Mono', '0.33 & 0.66 Poly', 
    '1.00 Mono', '1.00 Poly'
]
print("Classification Report:")
print(classification_report(y_test_true, y_test_pred, target_names=target_names, zero_division=0))

print("Confusion Matrix:")
cm = confusion_matrix(y_test_true, y_test_pred)
print(cm)

torch.save(model.state_dict(), "solar_cnn_weights.pth")
print("Saved Model Weights to 'solar_cnn_weights.pth'")

config_data = {
    "best_th_medium": float(best_th_medium),
    "best_th_high": float(best_th_high)
}

with open("app_config.json", "w") as f:
    json.dump(config_data, f)
print("Saved App Configuration Metadata to 'app_config.json'")

Using Device: cpu
Epoch: [01/200] Training Loss: 1.0486 | Validation Loss: 0.9782 | LR: 1.00e-03
Epoch: [02/200] Training Loss: 0.9135 | Validation Loss: 1.2032 | LR: 1.00e-03
Epoch: [03/200] Training Loss: 0.8703 | Validation Loss: 1.0573 | LR: 1.00e-03
Epoch: [04/200] Training Loss: 0.8574 | Validation Loss: 0.9146 | LR: 1.00e-03
Epoch: [05/200] Training Loss: 0.8096 | Validation Loss: 0.8655 | LR: 1.00e-03
Epoch: [06/200] Training Loss: 0.8079 | Validation Loss: 0.8277 | LR: 1.00e-03
Epoch: [07/200] Training Loss: 0.8332 | Validation Loss: 0.8545 | LR: 1.00e-03
Epoch: [08/200] Training Loss: 0.7990 | Validation Loss: 0.8149 | LR: 1.00e-03
Epoch: [09/200] Training Loss: 0.7773 | Validation Loss: 0.8900 | LR: 1.00e-03
Epoch: [10/200] Training Loss: 0.7593 | Validation Loss: 0.8371 | LR: 1.00e-03
Epoch: [11/200] Training Loss: 0.7641 | Validation Loss: 0.7893 | LR: 1.00e-03
Epoch: [12/200] Training Loss: 0.7538 | Validation Loss: 0.8031 | LR: 1.00e-03
Epoch: [13/200] Training Loss: 0.7